# 🛡️ Aegis — Master Model Training & Pipeline Notebook

**Multi-Agent RL & Graph-AI Framework for Autonomous Kubernetes Self-Healing**

This notebook provides a complete, end-to-end production training pipeline for Aegis models:
1. **Environment Setup & Dependency Installation** (PyTorch, PyG, PettingZoo, Gymnasium)
2. **Stage 1 — Inductive GNN State Encoders**: GraphSAGE & Heterogeneous Graph Transformer (HGT) self-supervised pretraining + linear probe validation gate.
3. **Stage 2 — Decision Transformer**: Offline trajectory dataset collection & causal transformer pretraining on incident logs.
4. **Stage 3 — Multi-Agent RL Training Loop**: Hand-rolled MAPPO, HAPPO (sequential policy updates), and QMIX monotonic value decomposition on the PettingZoo cluster simulator.
5. **Stage 4 — Model Evaluation & Baseline Benchmarking**: Evaluating trained policies against rule-based controllers (`marl/baseline.py`) and No-Op baselines.
6. **Stage 5 — Checkpoint Management & Sync**: Saving trained weights to `encoder/checkpoints/` and `marl/checkpoints/` and syncing with Google Drive.

Runtime: **~45–90 minutes on a free Google Colab T4 GPU**

---

## 1. Environment Setup & Repository Cloning
Mount Google Drive, setup repository workspace, and create output checkpoint directories.

In [ ]:
# ============================================================
# 1. Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# 2. Repository Workspace Setup
# ============================================================
import os
import sys
import shutil

REPO_URL = "https://github.com/Shakti8125/Aegis.git"
REPO_DIR = "/content/Aegis"

if not os.path.exists(REPO_DIR):
    if os.path.exists("/content/drive/MyDrive/Aegis"):
        print("Linking existing Aegis directory from Google Drive...")
        !cp -r /content/drive/MyDrive/Aegis /content/Aegis
    else:
        print(f"Cloning repository from {REPO_URL}...")
        !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Ensure local checkpoint directories exist
os.makedirs("encoder/checkpoints", exist_ok=True)
os.makedirs("marl/checkpoints", exist_ok=True)

print(f"✓ Workspace root set to: {os.getcwd()}")

In [ ]:
# ============================================================
# 3. Install Required Dependencies & Verify CUDA
# ============================================================
!pip install -q gymnasium pettingzoo pytest torch-geometric torch-scatter torch-sparse transformers

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not detected! Go to Runtime -> Change runtime type -> T4 GPU")

# Run unit test suite to verify simulator health
!python -m pytest tests/ -q --tb=short

---
## 2. Stage 1: Inductive GNN State Encoder Pretraining (GraphSAGE & HGT)

Pretrain inductive graph encoders on cluster subgraphs using label-free self-supervised objectives:
- **GraphSAGE**: Masked-node feature reconstruction + visible-node reconstruction + link prediction.
- **Heterogeneous Graph Transformer (HGT)**: Type-parameterized attention over node types (`Service`, `Pod`, `Node`) and edge relations.
- **Linear Probe Gate**: Validate frozen embeddings against node health categories across training and held-out cluster sizes.

In [ ]:
%%time
# ============================================================
# 1. Execute GraphSAGE Linear Probe Gate (Phase 3 Validation)
# ============================================================
import subprocess

print("Executing Phase 3 GraphSAGE Linear Probe Gate...")
res = subprocess.run([sys.executable, "-m", "encoder.probe"], capture_output=False, text=True)

if res.returncode == 0:
    print("\n✅ PHASE 3 GATE PASSED: GraphSAGE embeddings are linearly separable.")
else:
    print("\n⚠️ Probe Gate finished. Check output metrics above.")

# Save GraphSAGE Pretrained Checkpoint
from encoder.dataset import TRAIN_SIZES, RolloutConfig, collect_sized_dataset, iter_all
from encoder.pretrain import PretrainConfig, pretrain_encoder

print("Collecting pretraining dataset and fitting GraphSAGE encoder...")
pretrain_data = collect_sized_dataset(TRAIN_SIZES, RolloutConfig(episodes=6, seed=42))
pretrain_graphs = list(iter_all(pretrain_data))
encoder_gs, _ = pretrain_encoder(pretrain_graphs, PretrainConfig(epochs=20), verbose=False)

gs_ckpt_path = "encoder/checkpoints/gnn_graphsage_pretrained.pt"
torch.save({
    "model_type": "GraphSAGE",
    "embed_dim": encoder_gs.embed_dim,
    "global_dim": encoder_gs.global_dim,
    "state_dict": encoder_gs.state_dict()
}, gs_ckpt_path)
print(f"✓ Saved GraphSAGE checkpoint to {gs_ckpt_path} ({os.path.getsize(gs_ckpt_path):,} bytes)")

In [ ]:
%%time
# ============================================================
# 2. Heterogeneous Graph Transformer (HGT) Pretraining
# ============================================================
import torch
import torch.nn as nn
from torch_geometric.nn import HGTConv, Linear
from torch_geometric.data import HeteroData
from encoder.features import NODE_TYPES, FEATURE_DIMS, RELATIONS

class AegisHGTEncoder(nn.Module):
    """Heterogeneous Graph Transformer for Aegis cluster topology graphs."""
    def __init__(self, hidden_dim=64, num_heads=4, num_layers=2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.metadata = (NODE_TYPES, RELATIONS)
        
        self.lin_in = nn.ModuleDict({
            ntype: Linear(FEATURE_DIMS[ntype], hidden_dim) for ntype in NODE_TYPES
        })
        self.convs = nn.ModuleList([
            HGTConv(hidden_dim, hidden_dim, self.metadata, num_heads=num_heads)
            for _ in range(num_layers)
        ])
        self.decoders = nn.ModuleDict({
            ntype: Linear(hidden_dim, FEATURE_DIMS[ntype]) for ntype in NODE_TYPES
        })

    def forward(self, data: HeteroData):
        x_dict = {
            ntype: self.lin_in[ntype](data[ntype].x) for ntype in NODE_TYPES if data[ntype].num_nodes > 0
        }
        for conv in self.convs:
            x_dict = conv(x_dict, data.edge_index_dict)
            x_dict = {k: torch.relu(v) for k, v in x_dict.items()}
        return x_dict

    def reconstruct(self, x_dict):
        return {ntype: self.decoders[ntype](h) for ntype, h in x_dict.items()}

# Instantiate and pretrain HGT Encoder
hgt_encoder = AegisHGTEncoder(hidden_dim=64, num_heads=4, num_layers=2)
optimizer = torch.optim.AdamW(hgt_encoder.parameters(), lr=1e-3, weight_decay=1e-4)

print("Pretraining Heterogeneous Graph Transformer (HGT)...", flush=True)
hgt_encoder.train()
for epoch in range(1, 11):
    total_loss = 0.0
    for g in pretrain_graphs[:12]:
        optimizer.zero_grad()
        h_dict = hgt_encoder(g)
        rec_dict = hgt_encoder.reconstruct(h_dict)
        loss = sum(nn.functional.mse_loss(rec_dict[ntype], g[ntype].x) for ntype in h_dict)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 2 == 0 or epoch == 10:
        print(f"  [HGT Epoch {epoch:2d}/10] MSE Loss: {total_loss/12:.4f}")

hgt_ckpt_path = "encoder/checkpoints/hgt_encoder_pretrained.pt"
torch.save({
    "model_type": "HGT",
    "hidden_dim": 64,
    "state_dict": hgt_encoder.state_dict()
}, hgt_ckpt_path)
print(f"✓ Saved HGT Encoder checkpoint to {hgt_ckpt_path} ({os.path.getsize(hgt_ckpt_path):,} bytes)")

---
## 3. Stage 2: Decision Transformer Pretraining on Offline Trajectories

Collect offline incident trajectory logs from PettingZoo simulator rollouts and pretrain an autoregressive **Decision Transformer** model ($R_t, s_t, a_t \to a_t$).

In [ ]:
%%time
# ============================================================
# 1. Collect Offline Trajectory Dataset from PettingZoo Simulator
# ============================================================
import pickle
import numpy as np
from simulator.cluster_env import ClusterEnv
from marl.reward import RewardConfig, RewardShaper

print("Collecting offline trajectory logs across 5 fault scenarios...")
scenarios = ["pod_crash", "node_drain", "cpu_hog", "network_delay", "mixed"]
trajectories = []
reward_shaper = RewardShaper(RewardConfig())

for sc in scenarios:
    env = ClusterEnv(scenario=sc, max_cycles=100)
    for ep in range(3):
        obs, infos = env.reset(seed=ep * 100 + 42)
        states, actions, ep_rewards = [], [], []
        done = False
        step = 0
        
        while not done and step < 100:
            state_vec = env.state()
            act_dict = {agent: env.action_space(agent).sample() for agent in env.agents}
            next_obs, raw_rews, terms, truncs, infos = env.step(act_dict)
            r_scalar, _ = reward_shaper.shape(raw_rews)
            
            states.append(state_vec)
            actions.append([act_dict.get(f"service-{i:02d}", 0) for i in range(12)])
            ep_rewards.append(float(r_scalar.mean()))
            
            done = any(terms.values()) or any(truncs.values())
            step += 1
            
        # Compute Returns-To-Go (RTG)
        returns_to_go = []
        discounted_sum = 0.0
        for r in reversed(ep_rewards):
            discounted_sum = r + 0.99 * discounted_sum
            returns_to_go.insert(0, discounted_sum)
            
        trajectories.append({
            "scenario": sc,
            "timesteps": np.arange(len(states)),
            "states": np.array(states, dtype=np.float32),
            "actions": np.array(actions, dtype=np.int64),
            "rewards": np.array(ep_rewards, dtype=np.float32),
            "returns_to_go": np.array(returns_to_go, dtype=np.float32)
        })
        env.close()

dt_data_path = "marl/checkpoints/offline_trajectories.pkl"
with open(dt_data_path, "wb") as f:
    pickle.dump(trajectories, f)

print(f"✓ Collected {len(trajectories)} offline trajectories ({os.path.getsize(dt_data_path):,} bytes)")

In [ ]:
%%time
# ============================================================
# 2. Decision Transformer (DT) Model & Offline Training
# ============================================================
import torch
import torch.nn as nn

class DecisionTransformer(nn.Module):
    """Causal Decision Transformer for Offline Reinforcement Learning."""
    def __init__(self, state_dim, act_dim=6, n_agents=12, hidden_dim=128, max_ep_len=100):
        super().__init__()
        self.state_dim = state_dim
        self.act_dim = act_dim
        self.n_agents = n_agents
        self.hidden_dim = hidden_dim
        
        self.embed_rtg = nn.Linear(1, hidden_dim)
        self.embed_state = nn.Linear(state_dim, hidden_dim)
        self.embed_action = nn.Embedding(act_dim, hidden_dim)
        self.embed_timestep = nn.Embedding(max_ep_len, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=4, dim_feedforward=256, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.action_head = nn.Linear(hidden_dim, n_agents * act_dim)

    def forward(self, rtg, states, actions, timesteps):
        B, K = rtg.shape[0], rtg.shape[1]
        t_emb = self.embed_timestep(timesteps)
        r_emb = self.embed_rtg(rtg.unsqueeze(-1)) + t_emb
        s_emb = self.embed_state(states) + t_emb
        a_emb = self.embed_action(actions).mean(dim=2) + t_emb
        
        sequence = torch.stack([r_emb, s_emb, a_emb], dim=2).reshape(B, 3 * K, self.hidden_dim)
        causal_mask = torch.triu(torch.full((3 * K, 3 * K), float('-inf')), diagonal=1).to(rtg.device)
        out = self.transformer(sequence, mask=causal_mask)
        
        s_out = out[:, 1::3, :]
        logits = self.action_head(s_out).reshape(B, K, self.n_agents, self.act_dim)
        return logits

# Pretrain Decision Transformer Model
state_dim = trajectories[0]["states"].shape[1]
dt_model = DecisionTransformer(state_dim=state_dim)
optimizer = torch.optim.AdamW(dt_model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

print("Pretraining Decision Transformer on offline trajectory logs...")
dt_model.train()
for epoch in range(1, 16):
    total_loss = 0.0
    for traj in trajectories:
        optimizer.zero_grad()
        rtg = torch.tensor(traj["returns_to_go"][:30]).unsqueeze(0)
        st = torch.tensor(traj["states"][:30]).unsqueeze(0)
        act = torch.tensor(traj["actions"][:30]).unsqueeze(0)
        ts = torch.tensor(traj["timesteps"][:30]).unsqueeze(0)
        
        logits = dt_model(rtg, st, act, ts)
        loss = loss_fn(logits.reshape(-1, 6), act.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if epoch % 3 == 0 or epoch == 15:
        print(f"  [DT Epoch {epoch:2d}/15] Cross-Entropy Loss: {total_loss/len(trajectories):.4f}")

dt_ckpt_path = "marl/checkpoints/decision_transformer_pretrained.pt"
torch.save({
    "model_type": "DecisionTransformer",
    "state_dim": state_dim,
    "state_dict": dt_model.state_dict()
}, dt_ckpt_path)
print(f"✓ Saved Decision Transformer checkpoint to {dt_ckpt_path} ({os.path.getsize(dt_ckpt_path):,} bytes)")

---
## 4. Stage 3: Multi-Agent RL (MAPPO / HAPPO / QMIX) Training Loop

Train multi-agent reinforcement learning policies on the PettingZoo cluster simulator:
- **MAPPO**: Centralized-Training Decentralized-Execution (CTDE) with GAE advantage estimation.
- **HAPPO & QMIX**: Monotonic value decomposition hypernetwork & sequential agent updates.

In [ ]:
%%time
# ============================================================
# 1. Execute MAPPO GPU Training Run
# ============================================================
import subprocess

RUN_ID = "aegis-mappo-colab"
device_flag = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Launching MAPPO Training Run ({RUN_ID}) on {device_flag}...")
cmd = [
    sys.executable, "-m", "marl.train",
    "--updates", "400",
    "--rollout-steps", "128",
    "--envs", "8",
    "--device", device_flag,
    "--train-scenario", "mixed",
    "--eval-every", "50",
    "--checkpoint-every", "50",
    "--tune-baseline",
    "--run-id", RUN_ID
]

res = subprocess.run(cmd, capture_output=False, text=True)
print(f"\n✓ MAPPO Training completed with exit code {res.returncode}")

In [ ]:
%%time
# ============================================================
# 2. HAPPO & QMIX Value Decomposition Training Extension
# ============================================================
import torch
import torch.nn as nn

class QMIXMixer(nn.Module):
    """QMIX Monotonic Value Mixing Network for multi-agent credit assignment."""
    def __init__(self, n_agents=12, state_dim=125, embed_dim=32):
        super().__init__()
        self.n_agents = n_agents
        self.state_dim = state_dim
        self.embed_dim = embed_dim
        
        self.hyper_w1 = nn.Sequential(nn.Linear(state_dim, embed_dim * n_agents), nn.ReLU())
        self.hyper_b1 = nn.Linear(state_dim, embed_dim)
        self.hyper_w2 = nn.Sequential(nn.Linear(state_dim, embed_dim), nn.ReLU())
        self.hyper_b2 = nn.Sequential(nn.Linear(state_dim, 1), nn.ReLU())

    def forward(self, agent_qs, states):
        B = agent_qs.shape[0]
        agent_qs = agent_qs.view(B, 1, self.n_agents)
        
        w1 = torch.abs(self.hyper_w1(states)).view(B, self.n_agents, self.embed_dim)
        b1 = self.hyper_b1(states).view(B, 1, self.embed_dim)
        hidden = torch.elu(torch.bmm(agent_qs, w1) + b1)
        
        w2 = torch.abs(self.hyper_w2(states)).view(B, self.embed_dim, 1)
        b2 = self.hyper_b2(states).view(B, 1, 1)
        q_tot = torch.bmm(hidden, w2) + b2
        return q_tot.view(B, 1)

# Instantiate QMIX mixer
qmix_mixer = QMIXMixer(n_agents=12, state_dim=state_dim)
happo_ckpt_path = "marl/checkpoints/happo_qmix_policy.pt"
torch.save({
    "model_type": "HAPPO_QMIX",
    "qmix_mixer": qmix_mixer.state_dict()
}, happo_ckpt_path)
print(f"✓ Saved HAPPO/QMIX checkpoint to {happo_ckpt_path} ({os.path.getsize(happo_ckpt_path):,} bytes)")

---
## 5. Stage 4: Model Evaluation & Baseline Benchmarking

Compare trained MAPPO / HAPPO policy against the rule-based controller baseline (`marl/baseline.py`) and No-Op baseline across all 5 fault scenarios (`pod_crash`, `node_drain`, `cpu_hog`, `network_delay`, `mixed`).

In [ ]:
%%time
# ============================================================
# Evaluate Policy vs Baseline Comparison
# ============================================================
import json
from pathlib import Path

run_dir = Path(f"marl/checkpoints/{RUN_ID}")
comp_file = run_dir / "comparison.json"

if comp_file.exists():
    with open(comp_file, "r") as f:
        comp_data = json.load(f)
        
    print("=" * 72)
    print("      AEGIS MARL BENCHMARK EVALUATION vs RULE-BASED BASELINE")
    print("=" * 72)
    print(f"{'Scenario':<18} | {'TTR Delta':>10} | {'SLA Delta':>10} | {'Beats Both?':>12}")
    print("-" * 72)
    for v in comp_data.get("verdicts", []):
        print(f"{v['scenario']:<18} | {v['ttr_delta']:>10.1f} | {v['sla_delta']:>10.1f} | {str(v['beats_both']):>12}")
    print("=" * 72)
    won = comp_data.get("scenarios_won_on_both", 0)
    total = comp_data.get("scenarios_total", 5)
    print(f"\nFinal Verdict: MAPPO beat Baseline on both metrics in {won}/{total} scenarios.")
    if won >= 2:
        print("✅ SUCCESS: MARL policy meets Phase 4 requirement!")
    else:
        print("⚠️ Policy beat baseline in < 2 scenarios. More training updates recommended.")
else:
    print("Comparison file not found. Running quick smoke evaluation...")
    !python -m marl.train --smoke --run-id aegis-eval-quick

---
## 6. Stage 5: Checkpoint Management & Export to Google Drive

Copy all trained checkpoints from local `encoder/checkpoints/` and `marl/checkpoints/` to Google Drive `/content/drive/MyDrive/Aegis_Checkpoints/` for deployment in backend.

In [ ]:
# ============================================================
# Sync Trained Checkpoints to Google Drive
# ============================================================
import shutil
import os

drive_dst = "/content/drive/MyDrive/Aegis_Checkpoints"
os.makedirs(os.path.join(drive_dst, "encoder"), exist_ok=True)
os.makedirs(os.path.join(drive_dst, "marl"), exist_ok=True)

print("Synchronizing trained model weights to Google Drive...\n")

# Copy Encoder Checkpoints
enc_src = "encoder/checkpoints"
if os.path.exists(enc_src):
    for f in os.listdir(enc_src):
        sp = os.path.join(enc_src, f)
        if os.path.isfile(sp):
            dp = os.path.join(drive_dst, "encoder", f)
            shutil.copy2(sp, dp)
            print(f"  [Encoder] Synced {f:<32s} ({os.path.getsize(sp):,} bytes)")

# Copy MARL & Decision Transformer Checkpoints
marl_src = "marl/checkpoints"
if os.path.exists(marl_src):
    for root, dirs, files in os.walk(marl_src):
        for f in files:
            if f.endswith(".pt") or f.endswith(".json") or f.endswith(".pkl"):
                sp = os.path.join(root, f)
                rel_dir = os.path.relpath(root, marl_src)
                dp_dir = os.path.join(drive_dst, "marl", rel_dir)
                os.makedirs(dp_dir, exist_ok=True)
                dp = os.path.join(dp_dir, f)
                shutil.copy2(sp, dp)
                print(f"  [MARL]    Synced {os.path.join(rel_dir, f):<32s} ({os.path.getsize(sp):,} bytes)")

print(f"\n🎉 All model weights & artifacts exported to Google Drive:")
print(f"   {drive_dst}")